In [1]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard

In [2]:
# Clear any logs from previous runs
!rm -rf ./logs/

'rm' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
from datetime import datetime
from packaging import version

import tensorflow as tf
from tensorflow import keras

import pandas as pd
tf.debugging.experimental.enable_dump_debug_info('./logs/',
                                                 tensor_debug_mode="FULL_HEALTH", 
                                                 circular_buffer_size=-1)
from keras import backend as K
import numpy as np

print("TensorFlow version: ", tf.__version__)
assert version.parse(tf.__version__).release[0] >= 2, \
    "This notebook requires TensorFlow 2.0 or above."

INFO:tensorflow:Enabled dumping callback in thread MainThread (dump root: ./logs/, tensor debug mode: FULL_HEALTH)
TensorFlow version:  2.21.0


In [4]:
load_ferrari_dataset = pd.read_csv('Ferrari_stock.csv')

load_ferrari_dataset.head()

df_dropped_firstcolumn = load_ferrari_dataset.drop(load_ferrari_dataset.columns[0], axis=1)
print(df_dropped_firstcolumn.head())

df_clean = df_dropped_firstcolumn.dropna()


       Close       High        Low       Open    Volume
0  50.845524  56.364575  50.845524  55.467844  22498800
1  52.463337  53.803811  51.492650  52.759165   4545100
2  52.121284  53.618915  52.019593  53.406289   1967600
3  50.864014  52.694452  50.420271  52.694452   1466300
4  49.782391  50.836283  45.631549  50.660632   5949200


In [5]:
data_size = len(df_dropped_firstcolumn)

print("Data size: ", data_size)
# 85% of the data is for training.
train_pct = 0.85

train_size = int(data_size * train_pct)

# We want to use as predictors the 'Open', 'High', 'Low' and 'Close' columns, so we will use those as our input data.
x = df_dropped_firstcolumn[['Open', 'High', 'Low', 'Close']]

# Generate the output data.
# y = We want to predict the volume of stocks traded, so we will use the 'Volume' column as our output data.
y = df_dropped_firstcolumn['Volume']

# Split into test and train pairs.
x_train, y_train = x[:train_size], y[:train_size]
x_test, y_test = x[train_size:], y[train_size:]

Data size:  2598


In [6]:
from sklearn.preprocessing import MinMaxScaler
logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)

# Scale inputs
scaler_x = MinMaxScaler()
x_train_scaled = scaler_x.fit_transform(x_train)
x_test_scaled = scaler_x.transform(x_test)

# Convert y to 2D NumPy arrays
y_train = y_train.to_numpy().reshape(-1, 1)
y_test = y_test.to_numpy().reshape(-1, 1)

# Scale outputs
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

# Build model
model = keras.models.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1),
])

# Compile model
model.compile(
    loss='mse',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['mae']
)

# Train
training_history = model.fit(
    x_train_scaled, y_train_scaled,
    validation_data=(x_test_scaled, y_test_scaled),
    epochs=50,
    batch_size=32,
    callbacks=[tensorboard_callback],
)

# Evaluate
loss = model.evaluate(x_test_scaled, y_test_scaled)
print("Test loss:", loss)

Epoch 1/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 0.0014 - mae: 0.0200 - val_loss: 5.9064e-04 - val_mae: 0.0154
Epoch 2/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 7.8705e-04 - mae: 0.0110 - val_loss: 8.5325e-04 - val_mae: 0.0224
Epoch 3/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 7.9173e-04 - mae: 0.0107 - val_loss: 4.9592e-04 - val_mae: 0.0120
Epoch 4/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 7.9488e-04 - mae: 0.0111 - val_loss: 7.5075e-04 - val_mae: 0.0199
Epoch 5/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 7.8876e-04 - mae: 0.0107 - val_loss: 6.2647e-04 - val_mae: 0.0165
Epoch 6/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 7.8652e-04 - mae: 0.0108 - val_loss: 7.6979e-04 - val_mae: 0.0204
Epoch 7/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 7.8135e-04 - mae: 0.0107 - val_loss: 5.3611e-04 - val_mae: 0.0133
Epoch 8/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 7.8321e-04 - mae: 0.0105 - val_loss: 4.7049e-04 - val_mae: 0.0111
Epoc

In [7]:
#http://localhost:6006

In [8]:
%tensorboard --logdir logs/

Reusing TensorBoard on port 6006 (pid 25076), started 0:38:20 ago. (Use '!kill 25076' to kill it.)

A brief overview of the visualizations created in this example and the dashboards (tabs in top navigation bar) where they can be found:

* Scalars show how the loss and metrics change with every epoch. You can use them to also track training speed, learning rate, and other scalar values. Scalars can be found in the Time Series or Scalars dashboards.
* Graphs help you visualize your model. In this case, the Keras graph of layers is shown which can help you ensure it is built correctly. Graphs can be found in the Graphs dashboard.
* Histograms and Distributions show the distribution of a Tensor over time. This can be useful to visualize weights and biases and verify that they are changing in an expected way. Histograms can be found in the Time Series or Histograms dashboards. Distributions can be found in the Distributions dashboard.

Breakdown of the Debugger Interface
The Debugger Dashboard on the Tensorboard consists of five main components:

* __Alerts:__ This top-left section contains a list of alert events detected by the debugger in the debug data from the instrumented TensorFlow program. Each alert indicates a certain anomaly that warrants attention. In our case, this section highlights 499 NaN/∞ events with a salient pink-red color. This confirms our suspicion that the model fails to learn because of the presence of NaNs and/or infinities in its internal tensor values.
* __Python Execution Timeline:__ This is the upper half of the top-middle section. It presents the full history of the eager execution of ops and graphs. Each box of the timeline is marked by the initial letter of the op or graph’s name. We can navigate this timeline by using the navigation buttons and the scrollbar above the timeline.
* __Graph Execution:__ Located at the top-right corner of the GUI, this section will be central to our debugging task. It contains a history of all the floating-type tensors computed inside graphs, i.e., the ones compiled by @tf-functions.
* __Stack Trace:__ The bottom-right section, shows the stack trace of the creation of every single operation on the graph.
* __Source Code:__ The bottom-left section, highlights the source code corresponding to each operation on the graph.
